In [ ]:
CREATE OR REFRESH STREAMING TABLE ${bronze_table_name}
CLUSTER BY AUTO
AS SELECT *, _metadata AS source_metadata 
FROM STREAM read_files('${source_file_path}')

In [ ]:
CREATE OR REFRESH STREAMING TABLE ${silver_table_name} TBLPROPERTIES('delta.feature.variantType-preview' = 'supported')
CLUSTER BY AUTO
AS SELECT split_part(source_metadata.file_path, "/", ${ingestion_date_depth}) as ingestion_date, split_part(source_metadata.file_path, "/", ${billing_period_depth}) as billing_period, *, try_parse_json(tags) AS parsedTags 
FROM stream(${bronze_table_name})

In [ ]:
CREATE OR REPLACE MATERIALIZED VIEW ${gold_table_name} TBLPROPERTIES('delta.feature.variantType-preview' = 'supported')
CLUSTER BY AUTO
AS SELECT date, meterName, meterCategory, meterSubCategory, meterRegion, subscriptionName, resourceGroupName, try_variant_get(parsedTags, '$.Vendor', 'string') as vendor, parsedTags,
  -- price: cost as Azure Cost Management reports it, in billing currency, before credits and taxes. Usage covered by a
  -- reservation, savings plan or Azure Databricks pre-purchase (DBCU) is 0 in Actuals (the Purchase row carries the cost)
  -- and the prorated benefit cost in Amortized. Set cost_column to `cost` for older exports that lack costInBillingCurrency.
  ${cost_column} AS price,
  -- contracted_cost: negotiated price before any reservation, savings plan or pre-purchase benefit (the previous price formula).
  quantity * unitPrice AS contracted_cost,
  quantity, unitPrice, effectivePrice, chargeType, pricingModel, reservationId, reservationName, productOrderId, benefitName, billingCurrency
FROM (
    SELECT *, RANK() OVER (PARTITION BY billing_period ORDER BY ingestion_date DESC) AS rank
    FROM ${silver_table_name}
  ) WHERE rank = 1